In [60]:
import os
import json
import pandas as pd
from pathlib import Path
from scipy.stats import shapiro, skew, kurtosis

os.chdir("C:/Users/Hassa/Downloads/github-repos/Projects/bias-testing/synthetic-data-generation")

In [61]:
import json
import pandas as pd
from pathlib import Path

def load_judged_conversations_to_df(directory: str = "judged") -> pd.DataFrame:
    """
    Load every judged conversation JSON file in `directory` into a single DataFrame,
    with `_meta` and `judge_scores` fields flattened into top-level columns.
    """
    directory = Path(directory)
    records = []
    failed = []

    for filepath in directory.glob("*.json"):
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Flatten _meta
            meta = data.pop("_meta", {})
            for k, v in meta.items():
                data[f"meta_{k}"] = v

            # Flatten judge_scores
            judge_scores = data.pop("judge_scores", {})
            for k, v in judge_scores.items():
                data[f"judge_{k}"] = v

            data["source_file"] = filepath.name
            records.append(data)

        except Exception as e:
            failed.append({"file": str(filepath), "error": str(e)})

    df = pd.DataFrame(records)

    if failed:
        print(f"Warning: {len(failed)} file(s) failed to load.")
        for f in failed[:5]:
            print(f"  {f['file']}: {f['error']}")
        if len(failed) > 5:
            print(f"  ...and {len(failed) - 5} more")

    print(f"Loaded {len(df)} judged conversations into DataFrame.")
    return df, failed


In [62]:
pd.set_option('display.max_rows', None)
df, failed_loads = load_judged_conversations_to_df("judged")

Loaded 256 judged conversations into DataFrame.


In [63]:
import pandas as pd
from scipy.stats import shapiro, skew, kurtosis

def check_normality(df, judge_cols, group_cols=None):
    """
    df: DataFrame with judge score columns and (optionally) grouping columns
    judge_cols: list of judge score column names
    group_cols: list of columns to group by (e.g. ['meta_race_ethnicity','meta_gender'])
    """
    results = []

    # Overall, per metric
    for col in judge_cols:
        data = df[col].dropna()
        if len(data) < 3:
            continue
        stat, p = shapiro(data)
        results.append({
            "group": "OVERALL", "metric": col, "n": len(data),
            "shapiro_stat": round(stat, 4), "p_value": round(p, 4),
            "normal_at_.05": p > 0.05,
            "skew": round(skew(data), 3), "kurtosis": round(kurtosis(data), 3)
        })

    # Per group, per metric
    if group_cols:
        for keys, sub in df.groupby(group_cols):
            group_label = "_".join(map(str, keys)) if isinstance(keys, tuple) else str(keys)
            for col in judge_cols:
                data = sub[col].dropna()
                if len(data) < 3:
                    continue
                stat, p = shapiro(data)
                results.append({
                    "group": group_label, "metric": col, "n": len(data),
                    "shapiro_stat": round(stat, 4), "p_value": round(p, 4),
                    "normal_at_.05": p > 0.05,
                    "skew": round(skew(data), 3), "kurtosis": round(kurtosis(data), 3)
                })

    return pd.DataFrame(results)

In [64]:
judge_cols = ['judge_verification_rigor',
       'judge_empathy_tone', 'judge_efficiency', 'judge_resolution_effort',
       'judge_policy_grounding', 'judge_escalation_appropriateness']

In [65]:
check_normality(df, judge_cols, group_cols=None)

,group,metric,n,shapiro_stat,p_value,normal_at_.05,skew,kurtosis
0,OVERALL,judge_verification_rigor,256,0.4948,0.0,False,-2.021,2.467
1,OVERALL,judge_empathy_tone,256,0.6456,0.0,False,-1.418,0.442
2,OVERALL,judge_efficiency,256,0.6174,0.0,False,-1.558,1.032
3,OVERALL,judge_resolution_effort,256,0.6401,0.0,False,-1.348,0.268
4,OVERALL,judge_policy_grounding,256,0.6474,0.0,False,-1.556,1.123
5,OVERALL,judge_escalation_appropriateness,256,0.4797,0.0,False,-1.843,1.569


In [72]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon

judge_cols = [
    "judge_verification_rigor", "judge_empathy_tone", "judge_efficiency",
    "judge_resolution_effort", "judge_policy_grounding", "judge_escalation_appropriateness"
]

# --- Counterfactual (matched-set) tests ---
# `meta_persona_id` marks rows generated from the SAME persona slot (identical
# scenario, template, and injected agent_quality_tier). Age, race, AND gender
# all vary WITHIN a persona, so every test below blocks on persona_id PLUS
# the two identity dimensions NOT being tested — holding everything except
# the factor of interest fixed.
#
# Reference/control groups: race -> white, gender -> male, age -> under_62.
# Every other level is compared against its control, never against every
# other level pairwise — this matches an audit-study design (minority vs.
# majority baseline) and keeps the Holm correction less severe than an
# all-pairs comparison would.

RACE_CONTROL = "white"
GENDER_CONTROL = "male"
AGE_CONTROL = "under_62"


def holm_bonferroni(p_values):
    """Holm step-down correction for a list/array of raw p-values."""
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = min((n - rank) * p_values[idx], 1.0)
        running_max = max(running_max, adj)
        adjusted[idx] = running_max
    return adjusted



def race_friedman_test(df, metric_cols, race_col="meta_race_ethnicity",
                        block_cols=("meta_persona_id", "meta_gender", "meta_age_bucket")):
    """
    Friedman test per metric: does the score differ across all 4 race groups
    at once, blocked on (persona_id, gender, age_bucket)? Each block
    contributes exactly one score per race, holding scenario/tier/gender/age
    fixed. Run this omnibus check before reading the vs-control breakdown.
    """
    races = sorted(df[race_col].unique())
    results = []
    for col in metric_cols:
        pivot = df.pivot_table(index=list(block_cols), columns=race_col, values=col)
        pivot = pivot.dropna(subset=races)
        if len(pivot) < 3:
            results.append({"metric": col, "n_blocks": len(pivot), "friedman_stat": None,
                             "p_value": None, "significant_at_.05": False})
            continue
        stat, p = friedmanchisquare(*[pivot[r].values for r in races])
        results.append({
            "metric": col, "n_blocks": len(pivot),
            "friedman_stat": round(stat, 3), "p_value": round(p, 5),
            "significant_at_.05": p < 0.05,
        })
    return pd.DataFrame(results)


def pairwise_wilcoxon_vs_control(df, metric_cols, factor_col, control, block_cols):
    """
    Generic counterfactual test: paired Wilcoxon signed-rank comparing every
    OTHER level of `factor_col` against a fixed `control` level, blocked on
    `block_cols` (persona_id plus every identity dimension EXCEPT factor_col,
    so only factor_col varies within each block). Holm-corrected within each
    metric across the (k-1) comparisons against the control.

    mean_diff_vs_control > 0 means that level scored HIGHER than control
    within the same matched blocks; < 0 means lower.
    """
    levels = sorted(l for l in df[factor_col].unique() if l != control)
    rows = []
    for col in metric_cols:
        pivot = df.pivot_table(index=list(block_cols), columns=factor_col, values=col)
        pivot = pivot.dropna(subset=levels + [control])
        for lvl in levels:
            diffs = pivot[lvl] - pivot[control]
            if (diffs == 0).all():
                stat, p = np.nan, 1.0
            else:
                stat, p = wilcoxon(pivot[lvl], pivot[control])
            rows.append({
                "metric": col, "factor": factor_col, "level": lvl, "control": control,
                "n_pairs": len(pivot), "mean_diff_vs_control": round(diffs.mean(), 3),
                "wilcoxon_stat": stat, "p_raw": p,
            })
    result_df = pd.DataFrame(rows)
    result_df["p_holm"] = result_df.groupby("metric")["p_raw"].transform(
        lambda p: holm_bonferroni(p.values)
    )
    result_df["significant_holm_.05"] = result_df["p_holm"] < 0.05
    return result_df.sort_values(["metric", "p_holm"])


In [71]:

# ## Non parametric test across multiple groups, when normality fails
# print("=== Race effect: Friedman test, blocked on (persona_id, gender, age_bucket) ===")
# race_friedman = race_friedman_test(df, judge_cols)
# print(race_friedman.to_string(index=False))


print(f"\n=== Race effect: each race vs control='{RACE_CONTROL}' (Holm-corrected paired Wilcoxon) ===")
race_vs_control = pairwise_wilcoxon_vs_control(
    df, judge_cols, factor_col="meta_race_ethnicity", control=RACE_CONTROL,
    block_cols=("meta_persona_id", "meta_gender", "meta_age_bucket"),
)
print(race_vs_control.to_string(index=False))


=== Race effect: each race vs control='white' (Holm-corrected paired Wilcoxon) ===
                          metric              factor    level control  n_pairs  mean_diff_vs_control  wilcoxon_stat    p_raw   p_holm  significant_holm_.05
                judge_efficiency meta_race_ethnicity    asian   white       64                -0.016           30.0 0.763025 1.000000                 False
                judge_efficiency meta_race_ethnicity    black   white       64                -0.047           49.0 0.498709 1.000000                 False
                judge_efficiency meta_race_ethnicity hispanic   white       64                -0.016           30.0 0.763025 1.000000                 False
              judge_empathy_tone meta_race_ethnicity    asian   white       64                -0.047           15.0 0.317311 0.951932                 False
              judge_empathy_tone meta_race_ethnicity    black   white       64                 0.000           10.5 1.000000 1.000000   

In [69]:
print(f"=== Gender effect: vs control='{GENDER_CONTROL}' (paired Wilcoxon, blocked on persona_id, race, age_bucket) ===")
gender_vs_control = pairwise_wilcoxon_vs_control(
    df, judge_cols, factor_col="meta_gender", control=GENDER_CONTROL,
    block_cols=("meta_persona_id", "meta_race_ethnicity", "meta_age_bucket"),
)
print(gender_vs_control.to_string(index=False))

=== Gender effect: vs control='male' (paired Wilcoxon, blocked on persona_id, race, age_bucket) ===
                          metric      factor  level control  n_pairs  mean_diff_vs_control  wilcoxon_stat    p_raw   p_holm  significant_holm_.05
                judge_efficiency meta_gender female    male      128                 0.070           80.0 0.101391 0.101391                 False
              judge_empathy_tone meta_gender female    male      128                 0.023           56.0 0.491297 0.491297                 False
judge_escalation_appropriateness meta_gender female    male      128                 0.008           50.5 0.897702 0.897702                 False
          judge_policy_grounding meta_gender female    male      128                 0.039          110.5 0.356192 0.356192                 False
         judge_resolution_effort meta_gender female    male      128                 0.148          127.0 0.045161 0.045161                  True
        judge_verificati

In [70]:
print(f"=== Age effect: vs control='{AGE_CONTROL}' (paired Wilcoxon, blocked on persona_id, race, gender) ===")
age_vs_control = pairwise_wilcoxon_vs_control(
    df, judge_cols, factor_col="meta_age_bucket", control=AGE_CONTROL,
    block_cols=("meta_persona_id", "meta_race_ethnicity", "meta_gender"),
)
print(age_vs_control.to_string(index=False))

=== Age effect: vs control='under_62' (paired Wilcoxon, blocked on persona_id, race, gender) ===
                          metric          factor   level  control  n_pairs  mean_diff_vs_control  wilcoxon_stat    p_raw   p_holm  significant_holm_.05
                judge_efficiency meta_age_bucket over_62 under_62      128                 0.055          122.5 0.245739 0.245739                 False
              judge_empathy_tone meta_age_bucket over_62 under_62      128                 0.008          121.0 0.841481 0.841481                 False
judge_escalation_appropriateness meta_age_bucket over_62 under_62      128                 0.039           35.5 0.477011 0.477011                 False
          judge_policy_grounding meta_age_bucket over_62 under_62      128                -0.055          157.0 0.250588 0.250588                 False
         judge_resolution_effort meta_age_bucket over_62 under_62      128                -0.008          239.0 0.854266 0.854266              

In [ ]:
# import numpy as np
# from scipy.stats import chi2, binomtest

# # --- Counterfactual tests for the BINARY `outcome` column ---
# # outcome is categorical (resolved/escalated), not continuous like the judge
# # scores, so Friedman/Wilcoxon don't apply. The binary-data analogs of the
# # same persona_id-blocked design are Cochran's Q (>2 matched groups, i.e. the
# # race omnibus check) and McNemar's test (2 matched groups — gender, age,
# # and each race vs. the white control) — same RACE_CONTROL/GENDER_CONTROL/
# # AGE_CONTROL convention as the judge-score tests above, different math for
# # 0/1 data. Block columns always include ALL THREE identity dimensions
# # except the one under test, since age/race/gender all vary within a
# # persona_id now.

# df["escalated"] = (df["outcome"] == "escalated").astype(int)


# def cochrans_q_test(df, outcome_col, group_col="meta_race_ethnicity",
#                      block_cols=("meta_persona_id", "meta_gender", "meta_age_bucket")):
#     """
#     Cochran's Q test: does a binary outcome differ across `group_col`
#     (k >= 2 levels), blocked so each block contributes exactly one 0/1
#     observation per group. Non-parametric analog of Friedman for binary data.
#     """
#     groups = sorted(df[group_col].unique())
#     pivot = df.pivot_table(index=list(block_cols), columns=group_col, values=outcome_col)
#     pivot = pivot.dropna(subset=groups)
#     X = pivot[groups].values.astype(float)
#     n, k = X.shape
#     col_sums = X.sum(axis=0)
#     row_sums = X.sum(axis=1)
#     grand = X.sum()
#     denom = k * grand - np.sum(row_sums**2)
#     if denom == 0:
#         return {"n_blocks": n, "k_groups": k, "Q_stat": np.nan, "p_value": np.nan, "significant_at_.05": False}
#     Q = (k - 1) * (k * np.sum(col_sums**2) - grand**2) / denom
#     p = 1 - chi2.cdf(Q, df=k - 1)
#     return {"n_blocks": n, "k_groups": k, "Q_stat": round(Q, 3), "p_value": round(p, 5),
#             "significant_at_.05": p < 0.05}


# def mcnemar_vs_control(df, outcome_col, factor_col, control, block_cols):
#     """
#     Generic counterfactual test for BINARY outcomes: exact McNemar comparing
#     every OTHER level of `factor_col` against a fixed `control` level,
#     blocked on `block_cols` (persona_id plus every identity dimension EXCEPT
#     factor_col). Holm-corrected across the (k-1) comparisons against control.
#     Binary-data analog of pairwise_wilcoxon_vs_control.
#     """
#     levels = sorted(l for l in df[factor_col].unique() if l != control)
#     rows = []
#     for lvl in levels:
#         pivot = df.pivot_table(index=list(block_cols), columns=factor_col, values=outcome_col)
#         pivot = pivot.dropna(subset=[lvl, control])
#         x = pivot[lvl].values.astype(int)
#         y = pivot[control].values.astype(int)
#         b = int(np.sum((x == 1) & (y == 0)))  # level=1, control=0 in same block
#         c = int(np.sum((x == 0) & (y == 1)))  # control=1, level=0 in same block
#         n_disc = b + c
#         p = 1.0 if n_disc == 0 else binomtest(min(b, c), n_disc, 0.5).pvalue
#         rows.append({
#             "factor": factor_col, "level": lvl, "control": control, "n_pairs": len(pivot),
#             "level_only": b, "control_only": c, "n_discordant": n_disc, "p_raw": p,
#         })
#     result_df = pd.DataFrame(rows)
#     result_df["p_holm"] = holm_bonferroni(result_df["p_raw"].values)
#     result_df["significant_holm_.05"] = result_df["p_holm"] < 0.05
#     return result_df.sort_values("p_holm")

In [ ]:
# print("=== Race effect on escalation rate: Cochran's Q, blocked on (persona_id, gender, age_bucket) ===")
# print(cochrans_q_test(df, "escalated"))

# print(f"\n=== Race effect on escalation rate: each race vs control='{RACE_CONTROL}' (Holm-corrected exact McNemar) ===")
# print(mcnemar_vs_control(
#     df, "escalated", factor_col="meta_race_ethnicity", control=RACE_CONTROL,
#     block_cols=("meta_persona_id", "meta_gender", "meta_age_bucket"),
# ).to_string(index=False))

In [ ]:
# print(f"=== Gender effect on escalation rate: vs control='{GENDER_CONTROL}' (exact McNemar, blocked on persona_id, race, age_bucket) ===")
# print(mcnemar_vs_control(
#     df, "escalated", factor_col="meta_gender", control=GENDER_CONTROL,
#     block_cols=("meta_persona_id", "meta_race_ethnicity", "meta_age_bucket"),
# ).to_string(index=False))

In [ ]:
# print(f"=== Age effect on escalation rate: vs control='{AGE_CONTROL}' (exact McNemar, blocked on persona_id, race, gender) ===")
# print(mcnemar_vs_control(
#     df, "escalated", factor_col="meta_age_bucket", control=AGE_CONTROL,
#     block_cols=("meta_persona_id", "meta_race_ethnicity", "meta_gender"),
# ).to_string(index=False))